# M3 experiments: build and verify M3 (no full training, no corruption)
Run order: inventory → **m3_build_verify** → train_m3 → corruption → analysis

Run on: Colab **L4 GPU**. About 35-50 min (15 min of it is the data copy), about 3-4 compute units.

1. Setup, read-only snapshot of the existing project files
2. Code pinned to `main@8f10e61`
3. M3 code written to `m3_experiments/code/` (older versions kept in `code/_previous/`)
4. Configs, frozen experiment plan, project note for the Nepal folder owner
5. **Unit tests**
6. **Parameter counts**: M1, M3, exploratory ablation
7. Data copy to local disk (as in the training notebooks) and a file check
8. **Real-data forward pass and loss** (Florence and training batches)
9. **L4 benchmark**: M3 at batch 8, 12, 16 (M1 and the ablation at 8 for reference)
10. **Smoke runs** (320 training crops, 1 epoch, plus a resume test) into `m3/smoke/`, never into `m3/checkpoints/`
11. Preservation check and summary

Writes only inside `MAAI 902 X 901 PROJECT/m3_experiments/`. The three full M3 runs, the ablation run and the corruption
experiments are not started here.

## 1. Setup

In [ ]:
# GPU check, Drive mount and project paths
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os, sys, json, time, hashlib, datetime, platform, shutil, math, subprocess, gc
from google.colab import drive
drive.mount("/content/drive")

D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
CK, EXTRAS, NEPAL = f"{D}/checkpoints", f"{D}/eval_extras", f"{D}/eval_extras/nepal_final"
EXP = f"{D}/m3_experiments"
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"
REPO, REPO_COMMIT, CODE = "/content/FloodVerify", "8f10e61", "/content/m3code"
# The inventory notebook must have created m3_experiments/ already
assert os.path.isdir(EXP), "m3_experiments/ not found: run the Stage 2 notebook first"

# out(): every path this notebook writes to is built here; anything outside m3_experiments/ raises
_EXP_REAL = os.path.realpath(EXP)
def out(*parts):
    p = os.path.realpath(os.path.join(EXP, *parts))
    if not (p == _EXP_REAL or p.startswith(_EXP_REAL + os.sep)):
        raise ValueError(f"refusing to write outside m3_experiments: {p}")
    return p

# Write JSON, converting numpy values to plain Python numbers
def jdump(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

# Size and modification time of every existing project file (read only)
def snapshot():
    snap = {}
    for name in sorted(os.listdir(D)):
        if name != "m3_experiments":
            snap[f"[top] {name}"] = os.path.getmtime(f"{D}/{name}")
    for root in (CK, EXTRAS, f"{D}/figures"):
        if os.path.isdir(root):
            for dp, _, fs in os.walk(root):
                for fn in fs:
                    st = os.stat(os.path.join(dp, fn))
                    snap[os.path.relpath(os.path.join(dp, fn), D)] = [st.st_size, st.st_mtime]
    return snap

# Snapshot before any work, and the folders this notebook writes to
BEFORE = snapshot()
for s in ["code/_previous", "configs", "m3/benchmark", "m3/smoke", "m3/checkpoints",
          "m3/exploratory_ablation/checkpoints", "logs"]:
    os.makedirs(out(s), exist_ok=True)
print("snapshot entries:", len(BEFORE))

name, memory.total [MiB], driver_version
NVIDIA L4, 23034 MiB, 580.82.07
Mounted at /content/drive
snapshot entries: 119


## 2. Repository pinned to `main@8f10e61` (installed editable, not modified)

In [ ]:
# Fresh clone of the repository, pinned to commit 8f10e61 and installed in editable mode (not modified)
%cd /content
!rm -rf /content/FloodVerify
!git clone -q https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
%cd /content/FloodVerify
!git checkout -q 8f10e61 && git log -1 --format='%h %cd %s'
!pip install -q -e . --no-deps jsonschema
%cd /content
# Make the repo and the new M3 code folder importable
sys.path.insert(0, "/content/FloodVerify/src")
os.makedirs(CODE, exist_ok=True)
sys.path.insert(0, CODE)

/content
/content/FloodVerify
8f10e61 Fri Nov 20 14:07:06 2026 +0400 Merge pull request #28 from aaarrvind/m5/generate-advisory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for floodsitrep (pyproject.toml) ... done
/content


## 3. M3 code
The next four cells write the M3 code files to the Colab disk (`/content/m3code/`). The cell after them copies the files
to `m3_experiments/code/` and records each file's SHA-256 in `code/MANIFEST.json`; the later notebooks copy the files back
from Drive and check them against those hashes. The code in these four cells is exactly the code that ran.

### `m3_model.py`: the M3 model and the exploratory ablation
- **`MSTAUNet` (M3).** A shared ResNet-34 encoder runs on the pre and post images. At strides 2, 4 and 8,
  `LocalTemporalFusion` combines pre, post and their difference with a 1x1 convolution (as in M1), then applies channel
  attention (squeeze-excitation) and a spatial gate driven by |post - pre|. At strides 16 and 32,
  `CrossTemporalAttention` lets post features attend to pre features (4 heads, 2D sine-cosine positions, a residual
  scale `gamma` that starts at zero), followed by local fusion. The repo's U-Net decoder is reused unchanged.
- **Two heads.** A building head (1 channel) and a damage head (4 channels). `combine_heads` turns them into 5-class
  log-probabilities: log P(background) = log sigmoid(-z_b) and log P(class c) = log sigmoid(z_b) + log_softmax(z_d)_c.
  Because the output is a proper log-probability, the repo's training loss, validation, metrics and evaluation code run
  without changes.
- **`decomposed_weighted_ce`.** The repo's weighted cross-entropy written as a building term plus a damage term that
  only uses ground-truth building pixels. The unit tests use it to show the two are exactly equal.
- **`ExploratoryDualHeadUNet`.** M1 unchanged except that its output is replaced by the same two heads. Exploratory,
  one seed: it separates the effect of the two heads from the effect of attention fusion.

In [ ]:
%%writefile /content/m3code/m3_model.py
"""M3 Multi-Scale Temporal Attention U-Net, and the EXPLORATORY two-head ablation.

New work (m3_experiments/). Reuses the repo's ResNet-34 encoder, U-Net decoder and conv_bn_relu
(floodsitrep.cv.models.siamese_unet) unchanged.

Interface identical to M0-M2: ``forward(pre, post) -> [B, 5, H, W]``. For the two-head models the five
channels are LOG-PROBABILITIES of (background, no damage, minor, major, destroyed), built from
    building head  z_b [B,1,H,W]  and  damage head  z_d [B,4,H,W]:
        log P(background) = log sigmoid(-z_b)
        log P(class c)    = log sigmoid(z_b) + log_softmax(z_d)_c        c = 1..4
softmax(log P) = P and argmax is unchanged, so the repo's training loss (weighted CE), validation,
metrics, building-level scoring, calibration and evidence export all work without modification.

Why the damage loss is masked to building pixels: with the repo's weighted CE on these log-probs,
a background pixel contributes  w_0 * -log sigmoid(-z_b)  (no z_d term), an ignored pixel (255)
contributes nothing, and a ground-truth building pixel of class c contributes
w_c * (-log sigmoid(z_b) - log_softmax(z_d)_c). All terms share the denominator sum_i w_{y_i} over
non-ignored pixels. ``decomposed_weighted_ce`` computes exactly these terms (unit-tested).

Models
- ``MSTAUNet`` (M3): shared ResNet-34 on pre and post (one pass of 2B, as M1). Fusion per scale:
    strides 2, 4, 8:  LocalTemporalFusion = concat(a, p, p-a) -> 1x1 conv-BN-ReLU (as M1)
                      -> channel attention (squeeze-excitation) -> spatial gate from |p-a|
    strides 16, 32:   CrossTemporalAttention (post queries attend to pre keys/values, 4 heads,
                      2D sin-cos positions, zero-initialised residual scale) then LocalTemporalFusion
  then the repo's U-Net decoder (unchanged) and the two heads.
- ``ExploratoryDualHeadUNet`` (EXPLORATORY ABLATION, one seed): M1 exactly (same fusion, decoder,
  parameter names) with only the output replaced by the two heads. Separates "two heads" from
  "attention fusion" in the M3 vs M1 comparison.
"""
import torch
import torch.nn.functional as F
from torch import nn

from floodsitrep.cv.models.siamese_unet import ENC_CHANNELS, DEC_CHANNELS, ResNetEncoder, UNetDecoder, conv_bn_relu

IGNORE = 255
M3_CLASS_WEIGHTS = [0.10, 0.52, 0.88, 0.75, 2.74]   # identical to M1 weighted CE (verified in Stage 2)


# ----------------------------------------------------------------------------- heads -> 5 classes
def combine_heads(zb, zd):
    """Building logit [B,1,H,W] and damage logits [B,4,H,W] -> 5-class log-probabilities [B,5,H,W].

    Computed outside autocast; half-precision inputs are promoted to float32 (float64 stays float64).
    """
    with torch.autocast(device_type=zb.device.type, enabled=False):
        if zb.dtype in (torch.float16, torch.bfloat16):
            zb = zb.float()
        if zd.dtype in (torch.float16, torch.bfloat16):
            zd = zd.float()
        log_bg = F.logsigmoid(-zb)
        log_dmg = F.logsigmoid(zb) + F.log_softmax(zd, dim=1)
        return torch.cat([log_bg, log_dmg], dim=1)


def decomposed_weighted_ce(zb, zd, target, weights=M3_CLASS_WEIGHTS, ignore_index=IGNORE):
    """The repo's weighted CE on combine_heads(zb, zd), written as building + damage terms.

    total = [ sum_i w_{y_i} * bce_i  +  sum_{i: y_i in 1..4} w_{y_i} * dmg_nll_i ] / sum_i w_{y_i}
    over pixels with y_i != ignore_index, where bce_i = -log sigmoid(z_b) for building pixels and
    -log sigmoid(-z_b) for background, and dmg_nll_i = -log_softmax(z_d)_{y_i - 1}.
    """
    zb = zb[:, 0]
    w = torch.as_tensor(weights, dtype=zb.dtype, device=zb.device)
    keep = target != ignore_index
    t = torch.where(keep, target, torch.zeros_like(target)).clamp(0, 4)
    wt = w[t] * keep.to(zb.dtype)
    bld = keep & (t >= 1)
    bce = torch.where(bld, -F.logsigmoid(zb), -F.logsigmoid(-zb))
    dmg_nll = -F.log_softmax(zd, dim=1).gather(1, (t - 1).clamp(min=0).unsqueeze(1)).squeeze(1)
    denom = wt.sum()
    building_term = (wt * bce).sum() / denom
    damage_term = (wt * dmg_nll * bld.to(zb.dtype)).sum() / denom
    return {"total": building_term + damage_term, "building_term": building_term,
            "damage_term": damage_term, "denominator": denom,
            "n_valid": int(keep.sum()), "n_building": int(bld.sum())}


def _trunk_decoder():
    """The repo's UNetDecoder with its 3x3 classification conv removed: returns the 16-channel map."""
    dec = UNetDecoder(n_classes=1)
    dec.head = nn.Identity()
    return dec


class _TwoHeads(nn.Module):
    """Building (1) and damage (4) 3x3 convs on the final decoder map, like the repo's single head."""

    def __init__(self, cin=DEC_CHANNELS[-1]):
        super().__init__()
        self.head_building = nn.Conv2d(cin, 1, 3, padding=1)
        self.head_damage = nn.Conv2d(cin, 4, 3, padding=1)


# ----------------------------------------------------------------------------- fusion modules
class LocalTemporalFusion(nn.Module):
    """concat(a, p, p - a) -> 1x1 conv-BN-ReLU (as M1) -> channel attention -> spatial change gate."""

    def __init__(self, c, se_ratio=8):
        super().__init__()
        self.fuse = conv_bn_relu(3 * c, c, k=1)
        hidden = max(c // se_ratio, 8)
        self.channel_att = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, hidden, 1),
                                         nn.ReLU(inplace=True), nn.Conv2d(hidden, c, 1), nn.Sigmoid())
        self.spatial_gate = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3), nn.Sigmoid())

    def forward(self, a, p):
        d = p - a
        x = self.fuse(torch.cat([a, p, d], 1))
        x = x * self.channel_att(x)
        ad = d.abs()
        g = self.spatial_gate(torch.cat([ad.mean(1, keepdim=True), ad.amax(1, keepdim=True)], 1))
        return x * (1 + g)   # gate can only emphasise changed areas, never suppress below the M1-style feature


def sincos_2d(h, w, c, device):
    """Fixed 2D sine-cosine position code [h*w, c] (c divisible by 4)."""
    if c % 4:
        raise ValueError("channels must be divisible by 4")
    cq = c // 4
    omega = 1.0 / (10000 ** (torch.arange(cq, device=device, dtype=torch.float32) / cq))
    y = torch.arange(h, device=device, dtype=torch.float32)[:, None] * omega[None]   # [h, cq]
    x = torch.arange(w, device=device, dtype=torch.float32)[:, None] * omega[None]   # [w, cq]
    pe = torch.cat([y.sin()[:, None].expand(h, w, cq), y.cos()[:, None].expand(h, w, cq),
                    x.sin()[None].expand(h, w, cq), x.cos()[None].expand(h, w, cq)], dim=-1)
    return pe.reshape(h * w, c)


class CrossTemporalAttention(nn.Module):
    """Post features query pre features (multi-head attention), residual with a learned scale
    initialised at 0 (training starts from the M1-style local fusion), then LocalTemporalFusion."""

    def __init__(self, c, heads=4, se_ratio=8):
        super().__init__()
        self.norm_q = nn.LayerNorm(c)
        self.norm_kv = nn.LayerNorm(c)
        self.attn = nn.MultiheadAttention(c, heads, batch_first=True)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.local = LocalTemporalFusion(c, se_ratio)

    def forward(self, a, p):
        b, c, h, w = p.shape
        pe = sincos_2d(h, w, c, p.device)
        q = self.norm_q(p.flatten(2).transpose(1, 2))
        kv = self.norm_kv(a.flatten(2).transpose(1, 2))
        pe = pe.to(q.dtype)
        att, _ = self.attn(q + pe, kv + pe, kv, need_weights=False)
        p2 = p + self.gamma * att.transpose(1, 2).reshape(b, c, h, w).to(p.dtype)
        return self.local(a, p2)


# ----------------------------------------------------------------------------- models
class MSTAUNet(_TwoHeads):
    """M3: Multi-Scale Temporal Attention U-Net (shared ResNet-34, attention fusion, two heads)."""

    ATTENTION_SCALES = (3, 4)   # indices into the 5 encoder scales: strides 16 and 32

    def __init__(self, n_classes=5, pretrained=True, heads=4, se_ratio=8):
        if n_classes != 5:
            raise ValueError("M3 combines 1 building + 4 damage outputs into exactly 5 classes")
        super().__init__()
        self.encoder = ResNetEncoder(in_channels=3, pretrained=pretrained)
        self.fuse = nn.ModuleList(
            CrossTemporalAttention(c, heads, se_ratio) if i in self.ATTENTION_SCALES
            else LocalTemporalFusion(c, se_ratio)
            for i, c in enumerate(ENC_CHANNELS))
        self.decoder = _trunk_decoder()

    def forward_heads(self, pre, post):
        b = pre.shape[0]
        feats = self.encoder(torch.cat([pre, post], 0))     # one pass, shared weights (as M1)
        fused = [fuse(f[:b], f[b:]) for f, fuse in zip(feats, self.fuse)]
        x = self.decoder(fused)
        return self.head_building(x), self.head_damage(x)

    def forward(self, pre, post):
        return combine_heads(*self.forward_heads(pre, post))


class ExploratoryDualHeadUNet(_TwoHeads):
    """EXPLORATORY ABLATION (one seed): M1 (SiameseUNet) with only its output replaced by two heads."""

    def __init__(self, n_classes=5, pretrained=True):
        if n_classes != 5:
            raise ValueError("two heads combine into exactly 5 classes")
        super().__init__()
        self.encoder = ResNetEncoder(in_channels=3, pretrained=pretrained)
        self.fuse = nn.ModuleList(conv_bn_relu(3 * c, c, k=1) for c in ENC_CHANNELS)   # = M1
        self.decoder = _trunk_decoder()

    def forward_heads(self, pre, post):
        b = pre.shape[0]
        feats = self.encoder(torch.cat([pre, post], 0))
        fused = []
        for f, fuse in zip(feats, self.fuse):
            a, p = f[:b], f[b:]
            fused.append(fuse(torch.cat([a, p, p - a], 1)))
        x = self.decoder(fused)
        return self.head_building(x), self.head_damage(x)

    def forward(self, pre, post):
        return combine_heads(*self.forward_heads(pre, post))

Writing /content/m3code/m3_model.py


### `m3_register.py`: adds the new models to the repo's registry at run time
`register()` wraps the repo's `build_model` so that it also knows `m3_msta_unet` and `exploratory_abl_dualhead_unet`;
every existing model name goes to the original function unchanged. The patch is made in memory only: no repo file is
edited, and `load_checkpoint` and `train.py` then work on the new models.

In [ ]:
%%writefile /content/m3code/m3_register.py
"""Adds M3 and the EXPLORATORY ablation to the repo's model registry at runtime. No repo file is edited.

    import m3_register; m3_register.register()

After this, floodsitrep.cv.models.build_model / load_checkpoint and floodsitrep.cv.train.build_model
also know the two new names; every existing name is passed to the original function untouched.
"""
import floodsitrep.cv.models as _models
import floodsitrep.cv.train as _train

_ORIGINAL = getattr(_models, "_m3_original_build_model", _models.build_model)
_models._m3_original_build_model = _ORIGINAL          # safe if this module is imported twice

NEW_MODELS = {
    "m3_msta_unet": "M3 Multi-Scale Temporal Attention U-Net",
    "exploratory_abl_dualhead_unet": "EXPLORATORY ABLATION: M1 fusion + two heads (one seed)",
}


def build_model(cfg, pretrained=None):
    name = cfg["model"]
    if name not in NEW_MODELS:
        return _ORIGINAL(cfg, pretrained)
    pre = cfg.get("pretrained", True) if pretrained is None else pretrained
    n = cfg.get("n_classes", 5)
    from m3_model import ExploratoryDualHeadUNet, MSTAUNet

    if name == "m3_msta_unet":
        return MSTAUNet(n_classes=n, pretrained=pre, heads=cfg.get("attn_heads", 4),
                        se_ratio=cfg.get("se_ratio", 8))
    return ExploratoryDualHeadUNet(n_classes=n, pretrained=pre)


def register():
    _models.build_model = build_model   # used by load_checkpoint (looked up in that module at call time)
    _train.build_model = build_model    # train.py imported the name directly
    return build_model

Writing /content/m3code/m3_register.py


### `train_m3.py`: training launcher
Registers the two new models, then runs the repo's own `floodsitrep/cv/train.py` unchanged (same optimiser, schedule,
mixed precision, gradient clipping, resume, validation and best-epoch rule as M0-M2). It refuses any `--out-dir` that
is not inside an `m3_experiments/` folder.

In [ ]:
%%writefile /content/m3code/train_m3.py
"""Launcher: the repo's own floodsitrep/cv/train.py, UNCHANGED, with M3 and the exploratory ablation registered.

    python train_m3.py --config <m3_experiments>/configs/m3_msta_unet_ce.yaml \
        --xbd-root /content/xbd --manifest /content/xbd/rscc_flood_manifest.csv \
        --out-dir "<m3_experiments>/m3/checkpoints" [--set seed=1 ...]

Same optimiser, schedule, AMP, gradient clipping, resume, validation, metrics and best-epoch rule as M0-M2.
Refuses any --out-dir that is not inside an m3_experiments/ folder.
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))


def _check_out_dir(argv):
    if "--out-dir" not in argv:
        return
    d = os.path.realpath(argv[argv.index("--out-dir") + 1])
    if "m3_experiments" not in d.split(os.sep):
        raise SystemExit(f"refusing: --out-dir {d} is not inside m3_experiments/")


if __name__ == "__main__":
    _check_out_dir(sys.argv[1:])
    import m3_register

    m3_register.register()
    from floodsitrep.cv import train

    raise SystemExit(train.main(sys.argv[1:]))

Writing /content/m3code/train_m3.py


### `m3_tests.py`: unit tests
`run_all(device)` runs eight tests and returns pass/fail with details: output shapes; the combined output is a
log-probability; the loss decomposition equals the training loss exactly (all class weights, ignored pixels,
weighted-mean denominator); gradient masking (the damage head gets no gradient from background or ignored pixels);
gradients reach the attention blocks; registry and checkpoint save/load round trip; M3 and the ablation share M1's
encoder and decoder structure; compatibility with the evaluation pipeline. Nothing is read from or written to Drive.

In [ ]:
%%writefile /content/m3code/m3_tests.py
"""Unit tests for M3 and the exploratory ablation (run in Colab; needs torch and floodsitrep).

    import m3_tests; results = m3_tests.run_all("cuda")

Each test returns (passed: bool, details: dict). Nothing here reads data or writes files except a
temporary checkpoint in the system temp folder (deleted afterwards).
"""
import os
import tempfile

import numpy as np
import torch
import torch.nn.functional as F

from m3_model import (IGNORE, M3_CLASS_WEIGHTS, ExploratoryDualHeadUNet, MSTAUNet, combine_heads,
                      decomposed_weighted_ce)

W = M3_CLASS_WEIGHTS


def _rand_heads(shape=(3, 17, 19), seed=0, scale=3.0, dtype=torch.float64, grad=False):
    g = torch.Generator().manual_seed(seed)
    b, h, w = shape
    zb = (torch.randn(b, 1, h, w, generator=g, dtype=dtype) * scale).requires_grad_(grad)
    zd = (torch.randn(b, 4, h, w, generator=g, dtype=dtype) * scale).requires_grad_(grad)
    return zb, zd, g


def _rand_target(shape, g, p_ignore):
    t = torch.randint(0, 5, shape, generator=g)
    t[torch.rand(shape, generator=g) < p_ignore] = IGNORE
    return t


# ------------------------------------------------------------------------------------------- tests
def test_shapes(device):
    det, ok = {}, True
    for cls in (MSTAUNet, ExploratoryDualHeadUNet):
        m = cls(pretrained=False).to(device).eval()
        for h, w in ((512, 512), (256, 384)):
            x = torch.randn(2, 3, h, w, device=device)
            with torch.no_grad():
                zb, zd = m.forward_heads(x, x.flip(-1))
                y = m(x, x.flip(-1))
            good = (tuple(zb.shape) == (2, 1, h, w) and tuple(zd.shape) == (2, 4, h, w)
                    and tuple(y.shape) == (2, 5, h, w) and y.dtype == torch.float32
                    and bool(torch.isfinite(y).all()))
            ok &= good
            det[f"{cls.__name__} {h}x{w}"] = (f"building {tuple(zb.shape)}, damage {tuple(zd.shape)} -> "
                                              f"combined {tuple(y.shape)} {str(y.dtype)} finite={bool(torch.isfinite(y).all())}")
        del m
    return ok, det


def test_combined_is_log_probability():
    zb, zd, _ = _rand_heads(scale=4.0)
    lp = combine_heads(zb, zd)
    pb, pd_ = torch.sigmoid(zb), torch.softmax(zd, 1)
    manual = torch.cat([1 - pb, pb * pd_], 1)
    sum_dev = (lp.exp().sum(1) - 1).abs().max().item()
    prob_dev = (lp.exp() - manual).abs().max().item()
    softmax_dev = (torch.softmax(lp, 1) - manual).abs().max().item()   # what CE / calibration apply
    argmax_same = bool(torch.equal(lp.argmax(1), manual.argmax(1)))
    ext = combine_heads(torch.tensor([60.0, -60.0]).view(2, 1, 1, 1), torch.tensor([[30.0, -30, 0, 5]] * 2).view(2, 4, 1, 1))
    ext_finite = bool(torch.isfinite(ext).all())
    half = combine_heads(zb.half(), zd.half())
    ok = sum_dev < 1e-12 and prob_dev < 1e-12 and softmax_dev < 1e-12 and argmax_same and ext_finite \
        and half.dtype == torch.float32
    return ok, {"max |sum P - 1| (fp64)": sum_dev, "max |P - manual| (fp64)": prob_dev,
                "max |softmax(logP) - P| (fp64)": softmax_dev, "argmax identical": argmax_same,
                "finite at logits +-60": ext_finite, "fp16 heads -> output dtype": str(half.dtype)}


def test_loss_decomposition():
    """Weighted CE exactly as train.py computes it, against the explicit two-head decomposition."""
    from floodsitrep.cv.models.losses import SegLoss

    wt64 = torch.tensor(W, dtype=torch.float64)
    cases = {"random, 20% ignored": 0.2, "random, none ignored": 0.0, "random, 50% ignored": 0.5,
             "background + ignored only": "bg", "destroyed only": "d4"}
    det, ok = {}, True
    for k, (name, spec) in enumerate(cases.items()):
        zb, zd, g = _rand_heads(seed=10 + k)
        shape = (3, 17, 19)
        if spec == "bg":
            t = torch.zeros(shape, dtype=torch.long)
            t[:, :4] = IGNORE
        elif spec == "d4":
            t = torch.full(shape, 4, dtype=torch.long)
        else:
            t = _rand_target(shape, g, spec)
        keep = t != IGNORE
        lp = combine_heads(zb, zd)
        ref64 = F.cross_entropy(lp, t, weight=wt64, ignore_index=IGNORE)
        train32 = SegLoss("ce", class_weights=W)(lp.float(), t)       # the exact call train.py makes
        dec = decomposed_weighted_ce(zb, zd, t, W)
        tc = t.clamp(0, 4)
        wy = wt64[tc][keep]
        nll = -lp.gather(1, tc.unsqueeze(1)).squeeze(1)[keep]
        explicit = (wy * nll).sum() / wy.sum()
        m = (~keep).unsqueeze(1)                                         # change logits at ignored pixels only
        zb2 = torch.where(m, zb + 100.0, zb)
        zd2 = torch.where(m.expand_as(zd), -7.0 * zd, zd)
        pert = F.cross_entropy(combine_heads(zb2, zd2), t, weight=wt64, ignore_index=IGNORE)
        present = sorted(int(c) for c in torch.unique(t[keep]))
        r = {"classes present": present, "pixels valid / ignored": f"{int(keep.sum())} / {int((~keep).sum())}",
             "train-loss call SegLoss('ce') fp32": float(train32), "F.cross_entropy fp64": float(ref64),
             "explicit sum(w*nll)/sum(w) fp64": float(explicit), "decomposed total fp64": float(dec["total"]),
             "  building term": float(dec["building_term"]), "  damage term": float(dec["damage_term"]),
             "denominator (decomposed)": float(dec["denominator"]), "denominator (sum w_y, non-ignored)": float(wy.sum()),
             "|decomposed - CE| fp64": abs(float(dec["total"] - ref64)),
             "|explicit - CE| fp64": abs(float(explicit - ref64)),
             "rel |SegLoss fp32 - CE fp64|": abs(float(train32) - float(ref64)) / float(ref64),
             "loss change when ignored-pixel logits altered": abs(float(pert - ref64))}
        case_ok = (r["|decomposed - CE| fp64"] < 1e-12 and r["|explicit - CE| fp64"] < 1e-12
                   and r["rel |SegLoss fp32 - CE fp64|"] < 1e-6
                   and r["loss change when ignored-pixel logits altered"] == 0.0
                   and abs(r["denominator (decomposed)"] - r["denominator (sum w_y, non-ignored)"]) < 1e-12)
        if spec == 0.2:
            case_ok &= present == [0, 1, 2, 3, 4]                       # all five weights exercised
        if spec == "bg":
            case_ok &= float(dec["damage_term"]) == 0.0
        r["PASS"] = case_ok
        ok &= case_ok
        det[name] = r
    return ok, det


def test_gradient_masking():
    """Damage head gets no gradient from background or ignored pixels; building head none from ignored."""
    zb, zd, g = _rand_heads(seed=3, grad=True)
    t = _rand_target((3, 17, 19), g, 0.2)
    F.cross_entropy(combine_heads(zb, zd), t, weight=torch.tensor(W, dtype=torch.float64),
                    ignore_index=IGNORE).backward()
    gd = zd.grad.abs().sum(1)
    gb = zb.grad[:, 0].abs()
    bg, ign, bld = t == 0, t == IGNORE, (t >= 1) & (t <= 4)
    det = {"max |dL/dz_d| on background px (fp64)": gd[bg].max().item(),
           "max |dL/dz_d| on ignored px": gd[ign].max().item(),
           "mean |dL/dz_d| on building px": gd[bld].mean().item(),
           "max |dL/dz_b| on ignored px": gb[ign].max().item(),
           "mean |dL/dz_b| on background px": gb[bg].mean().item()}
    # through a whole model in float32: an all-background batch gives the damage head (almost) no gradient
    torch.manual_seed(0)
    m = MSTAUNet(pretrained=False).train()
    x = torch.randn(2, 3, 128, 128)
    tgt = torch.zeros(2, 128, 128, dtype=torch.long)
    F.cross_entropy(m(x, x + 0.1), tgt, weight=torch.tensor(W), ignore_index=IGNORE).backward()
    gdh = m.head_damage.weight.grad.abs().max().item()
    gbh = m.head_building.weight.grad.abs().max().item()
    det["model fp32, all-background batch: max|grad damage head| / max|grad building head|"] = gdh / gbh
    ok = (det["max |dL/dz_d| on background px (fp64)"] < 1e-12 and det["max |dL/dz_d| on ignored px"] == 0.0
          and det["max |dL/dz_b| on ignored px"] == 0.0 and det["mean |dL/dz_d| on building px"] > 1e-6
          and det["mean |dL/dz_b| on background px"] > 1e-6 and gdh / gbh < 1e-4)
    return ok, det


def test_attention_gradients():
    torch.manual_seed(1)
    m = MSTAUNet(pretrained=False).train()
    x, y = torch.randn(2, 3, 128, 128), torch.randn(2, 3, 128, 128)
    tgt = torch.randint(0, 5, (2, 128, 128))

    def step():
        m.zero_grad(set_to_none=True)
        F.cross_entropy(m(x, y), tgt, weight=torch.tensor(W), ignore_index=IGNORE).backward()

    step()
    nonfinite = [n for n, p in m.named_parameters() if p.grad is not None and not torch.isfinite(p.grad).all()]
    zero = [n for n, p in m.named_parameters() if p.grad is None or float(p.grad.abs().max()) == 0.0]
    gam = {f"fuse.{i}.gamma grad": float(m.fuse[i].gamma.grad) for i in MSTAUNet.ATTENTION_SCALES}
    with torch.no_grad():
        for i in MSTAUNet.ATTENTION_SCALES:
            m.fuse[i].gamma.fill_(0.5)
    step()
    att_zero_after = [n for n, p in m.named_parameters() if ".attn." in n and float(p.grad.abs().max()) == 0.0]
    expected_zero = all(".attn." in n or ".norm_q." in n or ".norm_kv." in n for n in zero)
    ok = not nonfinite and all(v != 0.0 for v in gam.values()) and not att_zero_after and expected_zero
    return ok, {"non-finite grads": nonfinite, **gam,
                "params with zero grad at init (expected: attention only, gamma=0)": zero,
                "attention params still zero-grad once gamma=0.5": att_zero_after}


def test_registry_and_checkpoint_roundtrip():
    import floodsitrep.cv.models as MM
    import floodsitrep.cv.train as TR
    import m3_register
    from floodsitrep.cv.models.siamese_unet import SiameseUNet

    m3_register.register()
    det, ok = {"train.build_model patched": TR.build_model is m3_register.build_model,
               "models.build_model patched": MM.build_model is m3_register.build_model}, True
    det["existing name 'siamese_unet' still builds SiameseUNet"] = isinstance(
        MM.build_model({"model": "siamese_unet", "pretrained": False}), SiameseUNet)
    x = torch.randn(1, 3, 128, 128)
    for name, cls in (("m3_msta_unet", MSTAUNet), ("exploratory_abl_dualhead_unet", ExploratoryDualHeadUNet)):
        cfg = {"model": name, "pretrained": True, "n_classes": 5}
        m = MM.build_model(cfg, pretrained=False).eval()
        fd, path = tempfile.mkstemp(suffix=".pt")
        os.close(fd)
        try:
            torch.save({"model": m.state_dict(), "config": cfg, "epoch": 0, "val": {}}, path)
            m2, cfg2 = MM.load_checkpoint(path, "cpu")                  # strict load, as every evaluation does
            with torch.no_grad():
                same = bool(torch.equal(m(x, x), m2(x, x)))
        finally:
            os.remove(path)
        det[f"{name}: builds {cls.__name__}, load_checkpoint round-trip identical"] = isinstance(m2, cls) and same
    ok = all(det.values())
    return ok, det


def test_architecture_relations():
    """Ablation = M1 except the output; M3 = M1 encoder and decoder (names and shapes)."""
    from floodsitrep.cv.models.siamese_unet import SiameseUNet

    m1 = {k: tuple(v.shape) for k, v in SiameseUNet(pretrained=False).state_dict().items()}
    ab = {k: tuple(v.shape) for k, v in ExploratoryDualHeadUNet(pretrained=False).state_dict().items()}
    m3 = {k: tuple(v.shape) for k, v in MSTAUNet(pretrained=False).state_dict().items()}
    m1_body = {k: v for k, v in m1.items() if not k.startswith("decoder.head.")}
    ab_body = {k: v for k, v in ab.items() if not k.startswith("head_")}
    same_body = m1_body == ab_body
    enc_dec = lambda d: {k: v for k, v in d.items() if k.startswith("encoder.") or k.startswith("decoder.blocks.")}
    same_enc_dec = enc_dec(m1) == enc_dec(m3)
    n = lambda d: int(sum(np.prod(s) for s in d.values()))
    return same_body and same_enc_dec, {
        "ablation body == M1 body (all tensors except the output layer)": same_body,
        "M3 encoder + decoder == M1 encoder + decoder": same_enc_dec,
        "M1 output layer tensors": n({k: v for k, v in m1.items() if k.startswith("decoder.head.")}),
        "two-head output tensors": n({k: v for k, v in ab.items() if k.startswith("head_")})}


def test_eval_pipeline_compatibility():
    from floodsitrep.cv.eval import metrics as M
    from floodsitrep.cv.eval.calibration import softmax as np_softmax

    zb, zd, g = _rand_heads(shape=(2, 32, 32), seed=5, dtype=torch.float32)
    lp = combine_heads(zb, zd)
    t = _rand_target((2, 32, 32), g, 0.1)
    cm = M.confusion(lp.argmax(1).numpy(), t.numpy())
    s = M.flat(M.summarise(cm))
    p = np_softmax(lp.permute(0, 2, 3, 1).reshape(-1, 5).double().numpy(), 1.133)   # temperature-scaled, as calibration does
    ok = cm.shape == (5, 5) and int(cm.sum()) == int((t != IGNORE).sum()) and np.isfinite(s["miou"]) \
        and np.allclose(p.sum(1), 1.0)
    return ok, {"confusion shape": cm.shape, "pixels counted": int(cm.sum()),
                "valid pixels": int((t != IGNORE).sum()), "metrics computed": sorted(s)[:4] + ["..."],
                "temperature softmax sums to 1": bool(np.allclose(p.sum(1), 1.0))}


def run_all(device="cuda"):
    tests = [("shapes (both heads, combined output)", lambda: test_shapes(device)),
             ("combined output is a log-probability", test_combined_is_log_probability),
             ("weighted CE decomposition = training loss", test_loss_decomposition),
             ("gradient masking", test_gradient_masking),
             ("attention gradients", test_attention_gradients),
             ("registry + load_checkpoint round-trip", test_registry_and_checkpoint_roundtrip),
             ("architecture relations to M1", test_architecture_relations),
             ("evaluation pipeline compatibility", test_eval_pipeline_compatibility)]
    out = []
    for name, fn in tests:
        try:
            passed, det = fn()
        except Exception as e:                       # a crash is a failure, recorded with its message
            passed, det = False, {"exception": repr(e)}
        out.append({"test": name, "passed": bool(passed), "details": det})
        print(f"{'PASS' if passed else 'FAIL'}  {name}", flush=True)
    return out

Writing /content/m3code/m3_tests.py


### Save the code to Drive
Copies each file to `m3_experiments/code/`. If a file with the same name but different content is already there, it is
moved to `code/_previous/` first, so nothing is overwritten.

In [ ]:
# Copy the code to m3_experiments/code/; a different older version is moved to code/_previous/ first.
def sha(p):
    return hashlib.sha256(open(p, "rb").read()).hexdigest()
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
# Record the SHA-256 of every file in MANIFEST.json
manifest = {"written": stamp, "repo_commit": REPO_COMMIT, "files": {}}
for fn in sorted(os.listdir(CODE)):
    if not fn.endswith(".py"):
        continue
    src, dst = f"{CODE}/{fn}", out("code", fn)
    if os.path.exists(dst) and sha(dst) != sha(src):
        shutil.move(dst, out("code", "_previous", f"{fn[:-3]}.{stamp}.py"))
    if not os.path.exists(dst):
        shutil.copy(src, dst)
    manifest["files"][fn] = sha(src)
jdump(manifest, out("code", "MANIFEST.json"))
print(json.dumps(manifest, indent=1))

{
 "written": "20261120_122547",
 "repo_commit": "8f10e61",
 "files": {
  "m3_model.py": "d00c1757de1703f9d3af33c1e58e4774f635aa50e101cf2ab79ef9c6ca2c3e1a",
  "m3_register.py": "dd9a56ff2f4ef6452b431ef815d10554d563b568bbd1652367aec57133bec174",
  "m3_tests.py": "255f60c5f628f3a5061a379364a85720351837ddfa06e47782768332dba05e56",
  "train_m3.py": "72bc6ee5c954b88c239925c6198d430e5372d52ae362da1f9f95800b41537b73"
 }
}


## 4. Configs, frozen plan, note for the Nepal folder owner
Writes the M3 and ablation training configs to `configs/` and checks that each differs from the repo's M1 weighted-CE
config only in the model name and the M3 architecture keys. Freezes the experiment plan in `configs/experiment_plan.json`
before any M3 training, and writes `NOTES_for_project_owners.md` about the missing `DONE` file in the Nepal folder.

In [ ]:
# M3 training config: the M1 weighted-CE recipe with the model name and M3 architecture keys added
M3_YAML = '''# M3: Multi-Scale Temporal Attention U-Net (two heads combined into 5-class log-probabilities).
# Recipe identical to configs/cv/m1_siamese_unet_ce.yaml (M1 weighted CE) at repo commit 8f10e61; only `model`
# (plus the two M3 architecture keys and num_workers) differs. Run with m3_experiments/code/train_m3.py.
model: m3_msta_unet
encoder: resnet34
pretrained: true
loss: ce
class_weights: [0.10, 0.52, 0.88, 0.75, 2.74]
epochs: 20
batch_size: 8
lr: 3.0e-4
weight_decay: 1.0e-4
warmup_steps: 200
amp: true
seed: 0
tile_size: 512
max_train_tiles: 0
max_val_tiles: 0
num_workers: 8
attn_heads: 4
se_ratio: 8
exploratory: false
'''
# Exploratory ablation config (seed 0 only)
ABL_YAML = '''# EXPLORATORY ABLATION (one seed, seed 0 only): M1 siamese U-Net (same fusion and decoder) with M3's two heads.
# Purpose: separate the effect of the two-head output from the effect of attention fusion. Not part of the
# 3-seed comparison; label it "exploratory" in every table and figure.
model: exploratory_abl_dualhead_unet
encoder: resnet34
pretrained: true
loss: ce
class_weights: [0.10, 0.52, 0.88, 0.75, 2.74]
epochs: 20
batch_size: 8
lr: 3.0e-4
weight_decay: 1.0e-4
warmup_steps: 200
amp: true
seed: 0
tile_size: 512
max_train_tiles: 0
max_val_tiles: 0
num_workers: 8
exploratory: true
'''
# Each config may differ from the repo's M1 CE config only in the allowed keys;
# an older, different version is moved to code/_previous/ before writing
import yaml
repo_m1 = yaml.safe_load(open(f"{REPO}/configs/cv/m1_siamese_unet_ce.yaml"))
for name, text in (("m3_msta_unet_ce.yaml", M3_YAML), ("exploratory_abl_m1fusion_dualhead_ce.yaml", ABL_YAML)):
    cfg = yaml.safe_load(text)
    differ = {k: (repo_m1.get(k), cfg.get(k)) for k in set(repo_m1) | set(cfg) if repo_m1.get(k) != cfg.get(k)}
    print(f"{name}: keys differing from M1 CE config ->", differ)
    allowed = {"model", "num_workers", "attn_heads", "se_ratio", "exploratory"}
    assert set(differ) <= allowed, f"unexpected recipe difference: {set(differ) - allowed}"
    p = out("configs", name)
    if os.path.exists(p) and open(p).read() != text:
        shutil.move(p, out("code", "_previous", f"{name}.{stamp}"))
    open(p, "w").write(text)

# The frozen experiment plan (recorded before any M3 training)
PLAN = {
  "frozen": stamp, "repo_commit": REPO_COMMIT,
  "models": {"M0 Dice+focal": "m0_early_fusion_unet_s{0,1,2}", "M1 weighted CE (final model)": "m1_siamese_unet_ce_s{0,1,2}",
             "M1 Dice+focal": "m1_siamese_unet_s{0,1,2}", "M2 SegFormer CE": "m2_siamese_segformer_ce_s{0,1,2}",
             "M3 MSTA U-Net CE (new)": "m3_msta_unet_ce_s{0,1,2}",
             "EXPLORATORY ablation, 1 seed": "exploratory_abl_m1fusion_dualhead_ce_s0"},
  "m3_training": "identical to M1 weighted CE; selection by best Florence xview2_pixel_score; Nepal never used for any M3 choice",
  "corruption": {"test_set": "Nepal (role test), 2,476 crops, 619 images; same crops for every condition and model",
                 "applied_to": "post image only, uint8, before normalisation; pre image, labels and the clean valid mask unchanged",
                 "conditions": {"clean": None, "gaussian_noise_sigma_DN": [10, 25, 50],
                                "gaussian_blur_sigma_px": [1.0, 2.0, 4.0], "jpeg_quality": [50, 25, 10]},
                 "seeding": "per crop and condition (md5 of crop id + condition); input SHA-256 logged per pass"},
  "baseline": "each checkpoint's own clean Nepal pass in the same session/environment; saved Nepal results only for reproducibility",
  "reproduction_tolerance": {"share_pixels_changed": 1e-4, "pixel_score": 0.002, "building_score": 0.002, "per_class_f1": 0.0005},
  "metrics": ["building-class IoU", "binary mIoU (background, building)", "damage macro-F1 (mean of 4, GT building pixels, "
              "background predictions counted as FN)", "damage F1 harmonic mean", "5-class mIoU", "xView2-style pixel score",
              "building-level score/recall/precision/damage F1", "per-class precision/recall/F1",
              "absolute and relative degradation vs own clean"],
}
p = out("configs", "experiment_plan.json")
if os.path.exists(p):
    shutil.move(p, out("code", "_previous", f"experiment_plan.{stamp}.json"))
jdump(PLAN, p)

# Note for the owner of the Nepal folder (written once)
NOTE = out("NOTES_for_project_owners.md")
if not os.path.exists(NOTE):
    open(NOTE, "w").write(
        "# Notes for the owners of existing folders (written by the m3_experiments notebooks; nothing outside\n"
        "# m3_experiments/ was changed)\n\n"
        "## eval_extras/nepal_final/ (owner of the Nepal final-test notebook)\n"
        "Found during the Stage 2 inventory (20 Nov 2026):\n"
        "- `DONE` is missing, so the notebook's run-once guard is not active. Re-running that notebook would overwrite\n"
        "  `decision.txt` with a new time stamp and re-score Nepal. Suggest adding `DONE` (or renaming the notebook).\n"
        "- `evidence_agreement_test.json` is missing: the Nepal evidence-agreement result is not recorded\n"
        "  (also listed as 'still to record' in the Nepal results PDF).\n"
        "- The 12 `cmp_*_test.npz` files are intact and all written after `decision.txt` (18 Nov 13:29).\n"
        "The m3_experiments notebooks only READ this folder and did not create or change any file in it.\n")
print(open(NOTE).read())

m3_msta_unet_ce.yaml: keys differing from M1 CE config -> {'model': ('siamese_unet', 'm3_msta_unet'), 'attn_heads': (None, 4), 'num_workers': (None, 8), 'exploratory': (None, False), 'se_ratio': (None, 8)}
exploratory_abl_m1fusion_dualhead_ce.yaml: keys differing from M1 CE config -> {'model': ('siamese_unet', 'exploratory_abl_dualhead_unet'), 'num_workers': (None, 8), 'exploratory': (None, True)}
# Notes for the owners of existing folders (written by the m3_experiments notebooks; nothing outside
# m3_experiments/ was changed)

## eval_extras/nepal_final/ (owner of the Nepal final-test notebook)
Found during the Stage 2 inventory (20 Nov 2026):
- `DONE` is missing, so the notebook's run-once guard is not active. Re-running that notebook would overwrite
  `decision.txt` with a new time stamp and re-score Nepal. Suggest adding `DONE` (or renaming the notebook).
- `evidence_agreement_test.json` is missing: the Nepal evidence-agreement result is not recorded
  (also listed as 'still to rec

## 5. Unit tests
Shapes of both heads and the combined output; the output is a proper log-probability; the weighted-CE decomposition
reproduces the training loss exactly (all five class weights, ignored pixels, weighted-mean denominator); gradient
masking; attention gradients; registry and checkpoint round-trip; relation to M1; evaluation-pipeline compatibility.

In [ ]:
# Load the M3 modules, register the new models and run the unit tests
import importlib, torch
import m3_model, m3_register, m3_tests
for m in (m3_model, m3_register, m3_tests):
    importlib.reload(m)
m3_register.register()
DEV = "cuda"
UNIT = m3_tests.run_all(DEV)
jdump(UNIT, out("m3", "benchmark", "unit_tests.json"))
for r in UNIT:
    print(f"\n=== {'PASS' if r['passed'] else 'FAIL'}: {r['test']}")
    print(json.dumps(r["details"], indent=1, default=str))
# Stop if any test failed
assert all(r["passed"] for r in UNIT), "a unit test failed: stop and paste the output above"

PASS  shapes (both heads, combined output)
PASS  combined output is a log-probability
PASS  weighted CE decomposition = training loss
PASS  gradient masking
PASS  attention gradients
PASS  registry + load_checkpoint round-trip
PASS  architecture relations to M1
PASS  evaluation pipeline compatibility

=== PASS: shapes (both heads, combined output)
{
 "MSTAUNet 512x512": "building (2, 1, 512, 512), damage (2, 4, 512, 512) -> combined (2, 5, 512, 512) torch.float32 finite=True",
 "MSTAUNet 256x384": "building (2, 1, 256, 384), damage (2, 4, 256, 384) -> combined (2, 5, 256, 384) torch.float32 finite=True",
 "ExploratoryDualHeadUNet 512x512": "building (2, 1, 512, 512), damage (2, 4, 512, 512) -> combined (2, 5, 512, 512) torch.float32 finite=True",
 "ExploratoryDualHeadUNet 256x384": "building (2, 1, 256, 384), damage (2, 4, 256, 384) -> combined (2, 5, 256, 384) torch.float32 finite=True"
}

=== PASS: combined output is a log-probability
{
 "max |sum P - 1| (fp64)": 3.3306690738754696e-

## 6. Parameter counts

In [ ]:
from floodsitrep.cv.models import build_model
# Split parameter counts into encoder, fusion, decoder and output head(s)
def groups(m):
    g = {"encoder": 0, "fusion": 0, "decoder": 0, "output head(s)": 0}
    for n, p in m.named_parameters():
        k = ("encoder" if n.startswith("encoder.") else "fusion" if n.startswith("fuse.") else
             "output head(s)" if n.startswith(("head_", "decoder.head.")) else "decoder")
        g[k] += p.numel()
    return g
# Build each model without pretrained weights and count its parameters
PARAMS = []
for label, name in (("M1 siamese U-Net (reference)", "siamese_unet"), ("M3 MSTA U-Net", "m3_msta_unet"),
                    ("EXPLORATORY ablation (M1 fusion + two heads)", "exploratory_abl_dualhead_unet")):
    m = build_model({"model": name, "pretrained": False})
    g = groups(m)
    PARAMS.append({"model": label, "total_M": round(sum(g.values()) / 1e6, 3), **{f"{k}_M": round(v / 1e6, 4) for k, v in g.items()},
                   "trainable_M": round(sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6, 3)})
    del m
import pandas as pd
PARAMS_DF = pd.DataFrame(PARAMS)
PARAMS_DF.to_csv(out("m3", "benchmark", "param_counts.csv"), index=False)
print(PARAMS_DF.to_string(index=False))

                                       model  total_M  encoder_M  fusion_M  decoder_M  output head(s)_M  trainable_M
                M1 siamese U-Net (reference)   25.496    21.2847    1.0588     3.1516            0.0007       25.496
                               M3 MSTA U-Net   26.902    21.2847    2.4654     3.1516            0.0007       26.902
EXPLORATORY ablation (M1 fusion + two heads)   25.496    21.2847    1.0588     3.1516            0.0007       25.496


## 7. Data to local disk (10-15 min, same commands as the training notebooks) and a file check

In [ ]:
# Extract the training data and copy the validation and test folders (skipped if already on the local disk)
if not os.path.exists(f"{X}/hold"):
    !mkdir -p /content/xbd
    !tar -xf "{D}/xbd_tars/train_images_labels_targets.tar" -C /content/xbd
    !tar -xf "{D}/xbd_tars/test_images_labels_targets.tar" -C /content/xbd
    !cp -r "{D}/xbd/hold" /content/xbd/
    !cp -r "{D}/xbd/tier3" /content/xbd/
    !cp "{D}/xbd/rscc_flood_manifest.csv" /content/xbd/
!ls /content/xbd
# Check that every training and Florence crop has its pre, post and target files
from floodsitrep.data.dataset import XbdTiles
from floodsitrep.data.tiling import source_paths
missing = {}
for role in ("train", "val"):
    ds = XbdTiles(X, MAN, role=role)
    miss = [r for r in ds.rows if not all(os.path.exists(source_paths(X, r)[k]) for k in ("pre", "post", "target"))]
    missing[role] = {"crops": len(ds), "missing": len(miss)}
print(missing)        # expect train 5488 / val 2184, missing 0
assert all(v["missing"] == 0 for v in missing.values())

hold  rscc_flood_manifest.csv  test  tier3  train
{'train': {'crops': 5488, 'missing': 0}, 'val': {'crops': 2184, 'missing': 0}}


## 8. Real-data forward pass and loss
One Florence (validation) batch and one training batch (8 crops each, fixed seed, no augmentation), through M1 (reference),
M3 and the ablation at initialisation (ImageNet encoder, untrained rest). Checks: output shape and dtype, finiteness,
log-probabilities summing to 1, `model(pre, post)` matching `combine_heads(forward_heads(...))` (two separate GPU passes,
so compared with a tolerance and the difference reported), the training loss call
against the two-head decomposition on real labels (including real no-data pixels), and a backward pass.

In [ ]:
import numpy as np, random
from torch.utils.data import default_collate
from floodsitrep.cv.train import prepare_target
from floodsitrep.cv.models.losses import SegLoss
from m3_model import combine_heads, decomposed_weighted_ce, M3_CLASS_WEIGHTS

# 8 crops picked with a fixed seed, no augmentation
def fixed_batch(role, n=8, seed=0):
    ds = XbdTiles(X, MAN, role=role, augment=False)
    idx = random.Random(seed).sample(range(len(ds)), n)
    return default_collate([ds[i] for i in idx])

# One Florence batch and one training batch
BATCHES = {"val (Florence)": fixed_batch("val"), "train": fixed_batch("train")}
loss_fn = SegLoss("ce", M3_CLASS_WEIGHTS)
REAL = []
# Each model at initialisation: forward pass, training-loss call and backward pass on both batches
for label, name in (("M1 (reference)", "siamese_unet"), ("M3", "m3_msta_unet"),
                    ("EXPLORATORY ablation", "exploratory_abl_dualhead_unet")):
    torch.manual_seed(0)
    model = build_model({"model": name, "pretrained": True}).to(DEV)
    for split, b in BATCHES.items():
        pre, post = b["pre"].to(DEV), b["post"].to(DEV)
        tgt = prepare_target(b, torch.device(DEV))
        r = {"model": label, "batch": split, "input": f"2 x {tuple(pre.shape)} float32",
             "ignored_px (no-data or >4)": int((tgt == 255).sum()),
             "px per class": [int((tgt == k).sum()) for k in range(5)]}
        model.train(); model.zero_grad(set_to_none=True)
        with torch.autocast("cuda"):
            outp = model(pre, post)
        loss = loss_fn(outp, tgt)
        loss.backward()
        r.update({"output": f"{tuple(outp.shape)} {outp.dtype}", "finite": bool(torch.isfinite(outp).all()),
                  "train-loss call (AMP)": float(loss),
                  "grads finite": all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)})
        # Two-head models only: compare model() with combine_heads(), and the loss with its decomposition
        if hasattr(model, "forward_heads"):
            model.eval()
            with torch.no_grad(), torch.autocast("cuda"):
                zb, zd = model.forward_heads(pre, post)
                full = model(pre, post)
            comb = combine_heads(zb, zd)
            dec = decomposed_weighted_ce(zb.float(), zd.float(), tgt, M3_CLASS_WEIGHTS)
            ref = loss_fn(comb, tgt)
            r.update({"max |sum P - 1|": float((comb.exp().sum(1) - 1).abs().max()),
                      "max |model() - combine_heads(forward_heads())|": float((full - comb).abs().max()),
                      "eval: loss call": float(ref), "eval: decomposed total": float(dec["total"]),
                      "eval: building term": float(dec["building_term"]), "eval: damage term": float(dec["damage_term"]),
                      "eval: rel diff loss vs decomposed": abs(float(ref) - float(dec["total"])) / float(ref),
                      "head shapes": f"{tuple(zb.shape)} {tuple(zd.shape)}"})
        REAL.append(r)
    del model; gc.collect(); torch.cuda.empty_cache()
# Save, print, and stop if any check failed
jdump(REAL, out("m3", "benchmark", "real_data_checks.json"))
for r in REAL:
    print(json.dumps(r, default=str))
assert all(r["finite"] and r["grads finite"] for r in REAL)
assert all(r["eval: rel diff loss vs decomposed"] < 1e-5 and r["max |model() - combine_heads(forward_heads())|"] < 1e-3
           for r in REAL if "eval: loss call" in r)

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 231MB/s]
/tmp/ipykernel_1240/596189057.py:31: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  "train-loss call (AMP)": float(loss),


{"model": "M1 (reference)", "batch": "val (Florence)", "input": "2 x (8, 3, 512, 512) float32", "ignored_px (no-data or >4)": 0, "px per class": [2048461, 44341, 2960, 1390, 0], "output": "(8, 5, 512, 512) torch.float16", "finite": true, "train-loss call (AMP)": 1.4422494173049927, "grads finite": true}
{"model": "M1 (reference)", "batch": "train", "input": "2 x (8, 3, 512, 512) float32", "ignored_px (no-data or >4)": 0, "px per class": [1962306, 56802, 0, 74043, 4001], "output": "(8, 5, 512, 512) torch.float16", "finite": true, "train-loss call (AMP)": 1.5228257179260254, "grads finite": true}
{"model": "M3", "batch": "val (Florence)", "input": "2 x (8, 3, 512, 512) float32", "ignored_px (no-data or >4)": 0, "px per class": [2048461, 44341, 2960, 1390, 0], "output": "(8, 5, 512, 512) torch.float32", "finite": true, "train-loss call (AMP)": 0.862497866153717, "grads finite": true, "max |sum P - 1|": 2.384185791015625e-07, "max |model() - combine_heads(forward_heads())|": 0.0, "eval: lo

## 9. L4 benchmark
Methodology (recorded with the results): real training batches from `XbdTiles(role="train", augment=True)`, 2 x
[B, 3, 512, 512] float32 inputs, AMP FP16, `cudnn.benchmark = True` (as in `train.py`), AdamW + GradScaler + gradient
clipping 1.0, exactly the training step of `train.py`. Each timed region is bracketed by `torch.cuda.synchronize()`.
- **GPU step**: batches already in host memory (no data wait); 3 warm-up steps excluded, 12 timed; includes the
  host-to-device copy.
- **Data loading only**: a fresh `DataLoader` (8 workers, pinned memory), first batch excluded (worker start-up), 20 timed.
- **End to end**: the real `DataLoader` feeding training steps; 3 warm-up steps, 20 timed; time blocked waiting for
  data reported separately.
- **Peak VRAM**: `torch.cuda.max_memory_allocated` / `max_memory_reserved` over the GPU-step measurement.
- **Inference**: eval mode, `no_grad`, AMP, same preloaded batches, 2 warm-up, 12 timed (to plan the corruption runs).
Training uses batch 8 (the approved recipe); batch 12 and 16 are measured for information only.

In [ ]:
# Benchmark set-up: real training batches with augmentation, as in train.py
from torch.utils.data import DataLoader
from floodsitrep.cv.train import param_groups, _worker_init
torch.backends.cudnn.benchmark = True
NW = min(8, max(2, (os.cpu_count() or 2) - 1))
TRAIN_DS = XbdTiles(X, MAN, role="train", augment=True)
W8 = M3_CLASS_WEIGHTS

# DataLoader with the training settings (shuffle, pinned memory, worker seeding)
def make_loader(bs, seed):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(TRAIN_DS, batch_size=bs, shuffle=True, drop_last=True, num_workers=NW, pin_memory=True,
                      worker_init_fn=_worker_init, generator=g, persistent_workers=False)

# One training step exactly as train.py does it (AMP, gradient scaling, clipping at 1.0)
def train_step(model, opt, scaler, lf, b):
    pre = b["pre"].to(DEV, non_blocking=True); post = b["post"].to(DEV, non_blocking=True)
    tgt = prepare_target(b, torch.device(DEV))
    with torch.autocast("cuda"):
        logits = model(pre, post)
    loss = lf(logits, tgt)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update()
    return loss.detach()

# Time stamp taken after waiting for the GPU to finish
def sync_time():
    torch.cuda.synchronize(); return time.perf_counter()

# Data loading speed on its own (first batch excluded)
def data_only(bs, n=20):
    it = iter(make_loader(bs, seed=7)); next(it)
    t0 = time.perf_counter()
    for _ in range(n):
        next(it)
    return (time.perf_counter() - t0) / n

# Full benchmark of one model and batch size: GPU step, peak memory, end-to-end speed, inference
def bench(label, name, bs, n_warm=3, n_meas=12, n_e2e=20):
    r = {"model": label, "batch_size": bs}
    model = opt = None
    try:
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        torch.manual_seed(0)
        model = build_model({"model": name, "pretrained": True}).to(DEV).train()
        opt = torch.optim.AdamW(param_groups(model, {"lr": 3e-4, "encoder_lr_mult": 1.0}), weight_decay=1e-4)
        scaler = torch.amp.GradScaler("cuda"); lf = SegLoss("ce", W8)
        it = iter(make_loader(bs, seed=0)); batches = [next(it) for _ in range(n_warm + n_meas)]; del it
        for b in batches[:n_warm]:
            train_step(model, opt, scaler, lf, b)
        t0 = sync_time()
        for b in batches[n_warm:]:
            train_step(model, opt, scaler, lf, b)
        r["gpu_s_per_step"] = (sync_time() - t0) / n_meas
        r["peak_alloc_GB"] = torch.cuda.max_memory_allocated() / 1e9
        r["peak_reserved_GB"] = torch.cuda.max_memory_reserved() / 1e9
        # End to end: the real DataLoader feeding training steps; time spent waiting for data recorded separately
        it = iter(make_loader(bs, seed=1))
        for _ in range(n_warm):
            train_step(model, opt, scaler, lf, next(it))
        t0, wait = sync_time(), 0.0
        for _ in range(n_e2e):
            tw = time.perf_counter(); b = next(it); wait += time.perf_counter() - tw
            train_step(model, opt, scaler, lf, b)
        r["e2e_s_per_step"] = (sync_time() - t0) / n_e2e
        r["data_wait_s_per_step"] = wait / n_e2e
        del it
        # Inference speed in eval mode
        model.eval()
        with torch.no_grad():
            for b in batches[:2]:
                with torch.autocast("cuda"):
                    model(b["pre"].to(DEV), b["post"].to(DEV))
            t0 = sync_time()
            for b in batches[n_warm:]:
                with torch.autocast("cuda"):
                    model(b["pre"].to(DEV), b["post"].to(DEV))
            r["infer_s_per_batch"] = (sync_time() - t0) / n_meas
        r["infer_ms_per_crop"] = 1000 * r["infer_s_per_batch"] / bs
        r["train_crops_per_s_e2e"] = bs / r["e2e_s_per_step"]
        r["bottleneck"] = "data loading" if r["data_wait_s_per_step"] > 0.2 * r["e2e_s_per_step"] else "GPU"
        r["status"] = "ok"
    except torch.cuda.OutOfMemoryError:
        r["status"] = "OOM"
    finally:
        del model, opt
        gc.collect(); torch.cuda.empty_cache()
    print(r, flush=True)
    return r

# Hardware and software environment, recorded with the results
dev_props = torch.cuda.get_device_properties(0)
ENVB = {"gpu": torch.cuda.get_device_name(0), "gpu_total_memory_GB": round(dev_props.total_memory / 1e9, 2),
        "driver": subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                                 capture_output=True, text=True).stdout.strip(),
        "torch": torch.__version__, "cuda (torch build)": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
        "python": platform.python_version(), "cpu_count": os.cpu_count(), "num_workers": NW,
        "input": "pre and post, each [B, 3, 512, 512] float32 (ImageNet-normalised); target [B, 512, 512]",
        "precision": "AMP FP16 autocast + GradScaler", "cudnn.benchmark": True,
        "methodology": "see the markdown above this cell; times via time.perf_counter() bracketed by torch.cuda.synchronize()"}
print(ENVB)
# Run the benchmarks: M3 at batch 8, 12 and 16; M1 and the ablation at batch 8
BENCH = {"environment": ENVB, "data_only_s_per_batch": {}, "runs": []}
for bs in (8, 16):
    BENCH["data_only_s_per_batch"][bs] = data_only(bs); print("data only, batch", bs, BENCH["data_only_s_per_batch"][bs])
for label, name, bss in (("M1 (reference)", "siamese_unet", (8,)), ("M3", "m3_msta_unet", (8, 12, 16)),
                         ("EXPLORATORY ablation", "exploratory_abl_dualhead_unet", (8,))):
    for bs in bss:
        BENCH["runs"].append(bench(label, name, bs))
# Projected training time per epoch at batch 8 (excluding validation)
steps = len(TRAIN_DS) // 8
for r in BENCH["runs"]:
    if r["status"] == "ok" and r["batch_size"] == 8:
        r["projected_train_min_per_epoch_bs8 (excl. validation)"] = steps * r["e2e_s_per_step"] / 60
jdump(BENCH, out("m3", "benchmark", "benchmark_l4.json"))
BENCH_DF = pd.DataFrame(BENCH["runs"])
BENCH_DF.to_csv(out("m3", "benchmark", "benchmark_l4.csv"), index=False)
print(BENCH_DF.round(4).to_string(index=False))

{'gpu': 'NVIDIA L4', 'gpu_total_memory_GB': 23.66, 'driver': '580.82.07', 'torch': '2.14.1+cu130', 'cuda (torch build)': '13.0', 'cudnn': 92800, 'python': '3.13.16', 'cpu_count': 12, 'num_workers': 8, 'input': 'pre and post, each [B, 3, 512, 512] float32 (ImageNet-normalised); target [B, 512, 512]', 'precision': 'AMP FP16 autocast + GradScaler', 'cudnn.benchmark': True, 'methodology': 'see the markdown above this cell; times via time.perf_counter() bracketed by torch.cuda.synchronize()'}
data only, batch 8 0.1169028335500002
data only, batch 16 0.23289093899999216
{'model': 'M1 (reference)', 'batch_size': 8, 'gpu_s_per_step': 0.22428502950000015, 'peak_alloc_GB': 4.10534144, 'peak_reserved_GB': 4.20478976, 'e2e_s_per_step': 0.22748063229998933, 'data_wait_s_per_step': 0.00033010980001790813, 'infer_s_per_batch': 0.0618986939166651, 'infer_ms_per_crop': 7.737336739583138, 'train_crops_per_s_e2e': 35.16782909874291, 'bottleneck': 'GPU', 'status': 'ok'}
{'model': 'M3', 'batch_size': 8, 'g

## 10. Smoke runs (mechanics only; never compared with anything)
The unmodified `train.py` through `code/train_m3.py`, 320 training crops and 96 Florence crops, into `m3/smoke/`
(separate from `m3/checkpoints/`; run names start with `smoke_`). M3: 1 epoch, then the same command with `epochs=2`
must print `resumed from ... at epoch 1` and add a second epoch. Ablation: 1 epoch (labelled exploratory).

In [ ]:
# Smoke runs go to m3/smoke/ only, with a small number of crops
SMOKE_DIR = out("m3", "smoke")
SMOKE_SET = "max_train_tiles=320 max_val_tiles=96 log_every=10 warmup_steps=20 num_workers=8"
# Launch train_m3.py as a subprocess and capture its output
def run_train(cfg_name, extra):
    cmd = (f'python {CODE}/train_m3.py --config "{out("configs", cfg_name)}" --xbd-root {X} --manifest {MAN} '
           f'--out-dir "{SMOKE_DIR}" --set {SMOKE_SET} {extra}')
    t0 = time.time()
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    log = p.stdout + p.stderr
    print(log[-2500:])
    return p.returncode, log, time.time() - t0

# M3 for 1 epoch, the same command again with epochs=2 (must resume), and the ablation for 1 epoch
SMOKE = {}
for key, cfg_name, run_name, extras in [
        ("M3 epoch 1", "m3_msta_unet_ce.yaml", "smoke_m3_msta_unet_ce_s0", "epochs=1"),
        ("M3 resume to epoch 2", "m3_msta_unet_ce.yaml", "smoke_m3_msta_unet_ce_s0", "epochs=2"),
        ("EXPLORATORY ablation epoch 1", "exploratory_abl_m1fusion_dualhead_ce.yaml",
         "smoke_exploratory_abl_m1fusion_dualhead_ce_s0", "epochs=1")]:
    if key != "M3 resume to epoch 2" and os.path.exists(f"{SMOKE_DIR}/{run_name}"):
        shutil.rmtree(f"{SMOKE_DIR}/{run_name}")       # smoke folders only (inside m3/smoke), so reruns start clean
    rc, log, secs = run_train(cfg_name, f"run_name={run_name} {extras}")
    open(out("logs", f"{run_name}__{key.replace(' ', '_')}.log"), "w").write(log)
    SMOKE[key] = {"run_name": run_name, "return_code": rc, "seconds": round(secs, 1),
                  "resumed": "resumed from" in log, "epoch lines": [l for l in log.splitlines() if l.startswith("epoch ")]}

# Check that the smoke checkpoints load and give valid outputs
from floodsitrep.cv.models import load_checkpoint
for key in ("M3 resume to epoch 2", "EXPLORATORY ablation epoch 1"):
    rd = f"{SMOKE_DIR}/{SMOKE[key]['run_name']}"
    s = SMOKE[key]
    s["files"] = sorted(os.listdir(rd))
    s["metrics_lines"] = sum(1 for _ in open(f"{rd}/metrics.jsonl"))
    m, cfg = load_checkpoint(f"{rd}/best.pt", DEV)
    b = BATCHES["val (Florence)"]
    with torch.no_grad(), torch.autocast("cuda"):
        o = m(b["pre"].to(DEV), b["post"].to(DEV))
    s["best.pt loads via load_checkpoint"] = type(m).__name__
    s["best.pt config model / run_name"] = f"{cfg['model']} / {cfg['run_name']}"
    s["best.pt forward"] = f"{tuple(o.shape)}, max |sum P - 1| {float((o.float().exp().sum(1) - 1).abs().max()):.2e}"
    del m
# Resume passes if the log says it resumed and metrics.jsonl has two epochs
SMOKE["M3 resume to epoch 2"]["resume ok"] = (SMOKE["M3 resume to epoch 2"]["resumed"]
                                              and SMOKE["M3 resume to epoch 2"]["metrics_lines"] == 2)
jdump(SMOKE, out("m3", "smoke", "smoke_summary.json"))
print(json.dumps(SMOKE, indent=1))

run smoke_m3_msta_unet_ce_s0: train 320 tiles, val 96 tiles, device cuda, amp True
  epoch 0 step 10/40 loss 1.4546 (running 1.1975)
  epoch 0 step 20/40 loss 1.1658 (running 1.1144)
  epoch 0 step 30/40 loss 1.1144 (running 1.0553)
  epoch 0 step 40/40 loss 0.7025 (running 1.0272)
epoch 0: train_loss 1.0272 val_loss 0.6149 xview2_pixel 0.0772 loc_f1 0.2573 dmg_f1 minor 0.000 major 0.213 destroyed 0.000 (39.6s)

run smoke_m3_msta_unet_ce_s0: train 320 tiles, val 96 tiles, device cuda, amp True
resumed from /content/drive/.shortcut-targets-by-id/1l_JXiqIVCCdRLWs8GOeTpO9VmYBoX8SY/MAAI 902 X 901 PROJECT/m3_experiments/m3/smoke/smoke_m3_msta_unet_ce_s0/last.pt at epoch 1 (best so far 0.0772)
  epoch 1 step 10/40 loss 0.8377 (running 0.9242)
  epoch 1 step 20/40 loss 0.9269 (running 0.8789)
  epoch 1 step 30/40 loss 1.2731 (running 0.8627)
  epoch 1 step 40/40 loss 1.2278 (running 0.8682)
epoch 1: train_loss 0.8682 val_loss 0.5238 xview2_pixel 0.1249 loc_f1 0.4162 dmg_f1 minor 0.000 major 0

## 11. Preservation check and summary

In [ ]:
# Nothing outside m3_experiments/ may have changed
AFTER = snapshot()
ch = {"added": sorted(set(AFTER) - set(BEFORE)), "removed": sorted(set(BEFORE) - set(AFTER)),
      "changed": sorted(k for k in set(BEFORE) & set(AFTER) if BEFORE[k] != AFTER[k])}
PRES = "NOTHING OUTSIDE m3_experiments CHANGED" if not any(ch.values()) else f"DIFFERENCES: {ch}"
jdump({"verdict": PRES, **ch}, out("m3", "benchmark", "preservation_check_stage3.json"))
# Summary of all checks, saved and printed
SUMMARY = {
  "unit_tests": {r["test"]: r["passed"] for r in UNIT},
  "decomposition (fp64, random 20% ignored)": {k: v for k, v in UNIT[2]["details"]["random, 20% ignored"].items()},
  "params": PARAMS, "real_data": REAL, "benchmark_env": ENVB,
  "benchmark": BENCH_DF.round(4).to_dict(orient="records"), "smoke": SMOKE, "preservation": PRES,
}
jdump(SUMMARY, out("m3", "benchmark", "stage3_summary.json"))
print(json.dumps(SUMMARY, indent=1, default=str))

{
 "unit_tests": {
  "shapes (both heads, combined output)": true,
  "combined output is a log-probability": true,
  "weighted CE decomposition = training loss": true,
  "gradient masking": true,
  "attention gradients": true,
  "registry + load_checkpoint round-trip": true,
  "architecture relations to M1": true,
  "evaluation pipeline compatibility": true
 },
 "decomposition (fp64, random 20% ignored)": {
  "classes present": [
   0,
   1,
   2,
   3,
   4
  ],
  "pixels valid / ignored": "803 / 166",
  "train-loss call SegLoss('ce') fp32": 4.674598217010498,
  "F.cross_entropy fp64": 4.674598488962188,
  "explicit sum(w*nll)/sum(w) fp64": 4.674598488962189,
  "decomposed total fp64": 4.6745984889621885,
  "  building term": 1.4346395647588457,
  "  damage term": 3.239958924203343,
  "denominator (decomposed)": 769.32,
  "denominator (sum w_y, non-ignored)": 769.3199999999999,
  "|decomposed - CE| fp64": 8.881784197001252e-16,
  "|explicit - CE| fp64": 1.7763568394002505e-15,
  "rel 

## End of notebook
Releases the Colab runtime so it stops using compute units.

In [ ]:
# Release the Colab runtime
from google.colab import runtime
runtime.unassign()